In [ ]:
#consolidate photos into one file.

import os
import shutil

# Source folders
source_folders = [
    "/Users/YOLO_training/C_East_6.27.23/bulk_images/training_images_C_East_6.27.23",
    "/Users/YOLO_training/C_East_6.27.23/bulk_images/training_images_C_East_6.27.23 2",
    "/Users/YOLO_training/C_East_6.27.23/bulk_images/training_images_C_East_6.27.23 3"
]

# Destination folder
destination_folder = "/Users/YOLO_training/C_East_6.27.23/images"

# Create destination if it doesn’t exist
os.makedirs(destination_folder, exist_ok=True)

# Choose behavior when a file with the same name already exists:
# "skip" = leave the duplicate behind
# "overwrite" = replace the existing file
duplicate_behavior = "skip"

for folder in source_folders:
    for filename in os.listdir(folder):
        src_path = os.path.join(folder, filename)
        dst_path = os.path.join(destination_folder, filename)

        if os.path.isfile(src_path):
            if os.path.exists(dst_path):
                if duplicate_behavior == "skip":
                    print(f"Skipped duplicate: {filename}")
                    continue
                elif duplicate_behavior == "overwrite":
                    print(f"Overwriting: {filename}")
                    os.remove(dst_path)

            shutil.move(src_path, dst_path)
            print(f"Moved: {filename}")

print("✅ All photos moved into:", destination_folder)

In [ ]:
#prep to sort photos

SRC      = Path("/Users/YOLO_training/C_East_6.27.23/images")   # folder containing all photos
TRAIN_DIR= Path("/Users/YOLO_training/C_East_6.27.23/images/train")    # existing train folder
VAL_DIR  = Path("/Users/YOLO_training/C_East_6.27.23/images/val")      # existing val folder

EXTS = {".png"}  # add/remove as needed
DRY_RUN = False      # set to False to actually move files
OVERWRITE = False   # if True, will overwrite files in target; if False, will skip conflicts

In [ ]:
#confirm existance of paths
assert SRC.exists() and SRC.is_dir(), f"Source not found: {SRC}"
assert TRAIN_DIR.exists() and TRAIN_DIR.is_dir(), f"Train dir not found: {TRAIN_DIR}"
assert VAL_DIR.exists() and VAL_DIR.is_dir(), f"Val dir not found: {VAL_DIR}"

In [ ]:
# Gather & shuffle photos, then spilt 80:20 train/validate.
images = [p for p in SRC.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
random.seed(42)
random.shuffle(images)
n = len(images)
assert n > 0, f"No images found in {SRC}"

n_train = max(1, round(0.8 * n))
train_imgs = images[:n_train]
val_imgs   = images[n_train:]

def move_list(files, dest):
    moved, skipped = 0, 0
    for src in files:
        tgt = dest / src.name
        if tgt.exists() and not OVERWRITE:
            skipped += 1
            print(f"SKIP (exists): {tgt}")
            continue
        if not DRY_RUN:
            if OVERWRITE and tgt.exists():
                tgt.unlink()
            shutil.move(src.as_posix(), tgt.as_posix())
        moved += 1
        print(("WOULD MOVE ->" if DRY_RUN else "MOVED ->"), src.name, "to", dest)
    return moved, skipped

print(f"Total images: {n} | Train: {len(train_imgs)} | Val: {len(val_imgs)} | DRY_RUN={DRY_RUN} | OVERWRITE={OVERWRITE}")

moved_tr, skipped_tr = move_list(train_imgs, TRAIN_DIR)
moved_val, skipped_val = move_list(val_imgs, VAL_DIR)

print(f"\nSummary:")
print(f" Train: moved={moved_tr} skipped={skipped_tr} -> {TRAIN_DIR}")
print(f" Val:   moved={moved_val} skipped={skipped_val} -> {VAL_DIR}")

In [ ]:
#show contents of zip folders to prep for extraction:
import zipfile
import os

zip_folder = "/Users/YOLO_training/C_East_6.27.23/labels/drive-download-20250909T023927Z-1-001"

for filename in os.listdir(zip_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(zip_folder, filename)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            file_list = zip_ref.namelist()
            print(f"\nContents of {filename}:")
            for f in file_list:
                print("  ", f)

In [ ]:
#pulls out file named annotations.xml and saves it next to the ZIP, renamed to match the ZIP’s base name.

import zipfile
import os
import shutil

# Folder containing your zip files
zip_folder = "path/to/your/folder"

for filename in os.listdir(zip_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(zip_folder, filename)

        # Base name without .zip
        base_name = os.path.splitext(filename)[0]

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for member in zip_ref.namelist():
                # Only target the annotations.xml file
                if member.endswith("annotations.xml"):
                    # Extract temporarily into the same folder
                    extracted_path = zip_ref.extract(member, zip_folder)

                    # New name = zip filename base + .xml
                    new_name = os.path.join(zip_folder, f"{base_name}.xml")

                    # Move/rename
                    shutil.move(extracted_path, new_name)
                    print(f"Extracted {member} from {filename} → {new_name}")

In [ ]:
#extract zip files, keep original names, leave in folder.

import zipfile
import os
import shutil

# Folder containing your zip files
zip_folder = "/Users/YOLO_training/C_East_6.27.23/labels/drive-download-20250909T023927Z-1-001"

for filename in os.listdir(zip_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(zip_folder, filename)

        # Base name without .zip
        base_name = os.path.splitext(filename)[0]

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for member in zip_ref.namelist():
                # Only target the annotations.xml file
                if member.endswith("annotations.xml"):
                    # Extract temporarily into the same folder
                    extracted_path = zip_ref.extract(member, zip_folder)

                    # New name = zip filename base + .xml
                    new_name = os.path.join(zip_folder, f"{base_name}.xml")

                    # Move/rename
                    shutil.move(extracted_path, new_name)
                    print(f"Extracted {member} from {filename} → {new_name}")

In [ ]:
#delete remaining zip files
import os

zip_folder = "/Users/YOLO_training/C_East_6.27.23/labels/drive-download-20250909T023927Z-1-001"

for filename in os.listdir(zip_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(zip_folder, filename)
        os.remove(zip_path)
        print(f"Deleted: {zip_path}")

In [ ]:
# creates txt files and saves classes.txt to help with next step

# --- CONFIG: edit these paths to your dataset layout ---
CVAT_XML_PATH = "/Users/YOLO_training/C_East_6.27.23/labels/drive-download-20250909T023927Z-1-001"  # file OR folder of XMLs
IMAGES_TRAIN  = "/Users/YOLO_training/C_East_6.27.23/images/train"  # images/train
IMAGES_VAL    = "/Users/YOLO_training/C_East_6.27.23/images/val"    # images/val
LABELS_TRAIN  = "/Users/YOLO_training/C_East_6.27.23/labels/train"  # labels/train
LABELS_VAL    = "/Users/YOLO_training/C_East_6.27.23/labels/val"    # labels/val

import xml.etree.ElementTree as ET       # parse XML
from pathlib import Path                 # paths
import os                                # misc

# --- make sure label folders exist ---
Path(LABELS_TRAIN).mkdir(parents=True, exist_ok=True)
Path(LABELS_VAL).mkdir(parents=True, exist_ok=True)

# --- collect image stems in each split (filename without extension) ---
IMG_EXT = {".png"}
def stems_in(folder):
    p = Path(folder)
    return {f.stem for f in p.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT} if p.exists() else set()

train_stems = stems_in(IMAGES_TRAIN)
val_stems   = stems_in(IMAGES_VAL)

# --- helpers: polygon -> tight bbox; bbox -> YOLO line ---
def polygon_to_bbox(points_str):
    # points_str like: "1061.27,563.22;1061.09,559.59;..."
    pts = []
    for token in points_str.replace(" ", "").split(";"):
        if not token:
            continue
        x_str, y_str = token.split(",")
        pts.append((float(x_str), float(y_str)))
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return min(xs), min(ys), max(xs), max(ys)  # (xtl, ytl, xbr, ybr)

def yolo_line(xtl, ytl, xbr, ybr, W, H, cls_id):
    x_c = ((xtl + xbr) / 2.0) / W
    y_c = ((ytl + ybr) / 2.0) / H
    w   = (xbr - xtl) / W
    h   = (ybr - ytl) / H
    return f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}"

# --- allow single XML file or a folder of XMLs ---
cvat_path = Path(CVAT_XML_PATH)
xml_files = [cvat_path] if cvat_path.is_file() else sorted(cvat_path.glob("*.xml"))
if not xml_files:
    raise FileNotFoundError(f"No XML files found at {CVAT_XML_PATH}")

# --- build/extend classes across ALL XMLs ---
name_to_id = {}
first_xml_dir = xml_files[0].parent  # for writing classes.txt at the end

n_images_total, n_written_total, n_skipped_total = 0, 0, 0

for CVAT_XML in xml_files:
    root = ET.parse(CVAT_XML).getroot()

    # use <meta><task><labels> if present to seed classes; extend if new labels appear
    label_nodes = root.findall("./meta/task/labels/label")
    if label_nodes:
        for ln in label_nodes:
            name = ln.findtext("name")
            if name is not None and name not in name_to_id:
                name_to_id[name] = len(name_to_id)

    # iterate images
    for img in root.findall(".//image"):
        n_images_total += 1
        fname = img.get("name")                         # e.g., "GH097515_0051.png"
        stem  = Path(fname).stem
        W     = float(img.get("width"))
        H     = float(img.get("height"))

        # decide split destination by membership
        if stem in train_stems:
            out_path = Path(LABELS_TRAIN) / f"{stem}.txt"
        elif stem in val_stems:
            out_path = Path(LABELS_VAL) / f"{stem}.txt"
        else:
            n_skipped_total += 1
            print(f"SKIP: {fname} not found in images/train or images/val")
            continue

        lines = []

        # handle <polygon>
        for poly in img.findall("polygon"):
            label = poly.get("label")
            if label not in name_to_id:
                name_to_id[label] = len(name_to_id)
            cls = name_to_id[label]
            xtl, ytl, xbr, ybr = polygon_to_bbox(poly.get("points"))
            # clip just in case
            xtl = max(0.0, min(xtl, W)); xbr = max(0.0, min(xbr, W))
            ytl = max(0.0, min(ytl, H)); ybr = max(0.0, min(ybr, H))
            if xbr > xtl and ybr > ytl:
                lines.append(yolo_line(xtl, ytl, xbr, ybr, W, H, cls))

        # write .txt (empty is fine)
        with open(out_path, "w") as f:
            f.write("\n".join(lines))
        n_written_total += 1

# --- save classes in id order once at the end ---
id_to_name = {i:n for n,i in name_to_id.items()}
ordered = [id_to_name[i] for i in range(len(id_to_name))]
with open(first_xml_dir / "classes.txt", "w") as f:
    f.write("\n".join(ordered))

print(f"Done. XML files processed: {len(xml_files)}, images in XMLs: {n_images_total}, labels written: {n_written_total}, skipped: {n_skipped_total}")
print("Classes (id order):", ordered)

In [ ]:
#Build a Ultralytics dataset.yaml from classes.txt (YOLOv8 compatible)
from pathlib import Path
import yaml


# EDIT ME: Set your dataset root folder (the folder that contains images/ and labels/)
DATA_ROOT = Path("/Users/YOLO_training/C_East_6.27.23")


# EDIT ME: Where is your classes.txt?

CLASSES_TXT = DATA_ROOT / "/Users/YOLO_training/C_East_6.27.23/labels/bulk_labels/classes.txt"

# EDIT ME (optional): Where to write the dataset.yaml
DATASET_YAML_OUT = Path("/Users/YOLO_training/C_East_6.27.23/labels/dataset.yaml")

# --- Sanity checks (helpful if paths are off) ---
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")

if not CLASSES_TXT.exists():
    raise FileNotFoundError(f"classes.txt not found at: {CLASSES_TXT}\n"
                            f"Make sure your XML→YOLO step produced it.")

# --- Read class names (one per line) ---
names = [ln.strip() for ln in CLASSES_TXT.read_text(encoding="utf-8").splitlines() if ln.strip()]
if not names:
    raise ValueError("No class names found in classes.txt (file is empty?)")

# --- (Optional) Verify expected subfolders exist ---
images_train = DATA_ROOT / "images" / "train"
images_val   = DATA_ROOT / "images" / "val"
labels_train = DATA_ROOT / "labels" / "train"
labels_val   = DATA_ROOT / "labels" / "val"

for p in [images_train, images_val, labels_train, labels_val]:
    if not p.exists():
        print(f"WARNING: Expected path not found: {p}")

# --- Build the YAML structure ---
# You can keep "train"/"val" relative to path (Ultralytics supports both relative and absolute).
yaml_dict = {
    "path": DATA_ROOT.as_posix(),          # Base path for relative entries below
    "train": "images/train",               # Relative to 'path' above (EDIT if your layout differs)
    "val":   "images/val",                 # Relative to 'path' above (EDIT if your layout differs)
    # Alternative: use absolute paths instead of relative:
    # "train": images_train.as_posix(),
    # "val": images_val.as_posix(),
    "names": {i: n for i, n in enumerate(names)}  # Explicit id->name mapping (robust + readable)
}

# --- Write YAML (human-friendly order preserved) ---
DATASET_YAML_OUT.write_text(yaml.safe_dump(yaml_dict, sort_keys=False), encoding="utf-8")

print("Wrote dataset YAML →", DATASET_YAML_OUT)
print("Classes:", names)
print("\nPreview YAML:\n" + DATASET_YAML_OUT.read_text(encoding="utf-8"))

In [ ]:
#time to run a training! Simple run with minimal edits, edit path to yaml if needed. change "projects" to control where output goes, and edit "name" as needed.
#NOTE: the yaml file contains the location of the train/val folders. change this in the yaml directly if you move stuff around.

# simple_yolo_train.py
import torch
from ultralytics import YOLO

# pick the best device available (M1/M2 -> "mps"; CUDA -> 0; else CPU)
device = "cpu" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else "cpu")

m = YOLO("yolov8n.pt")
m.train(
    data="/Users/YOLO_training/C_East_6.27.23/C_East_6.27.23.yaml",
    epochs=50,
    imgsz=640,
    batch=10,
    workers=0,
    device=device,
    mosaic=0.0, mixup=0.0,
    close_mosaic=0,
    val=True,
    cache=False,
    seed=0, deterministic=True,
    project="/Users/YOLO_training/runs",
    name="simple_n640_e20_b9")